In [1]:
!pip install fastapi httpx -q
import asyncio
import time
from typing import List
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from fastapi.testclient import TestClient

app = FastAPI()


class Request(BaseModel):
    numbers: List[int]
    delays: List[float]


class Result(BaseModel):
    number: int
    square: int
    delay: float
    time: float


class Response(BaseModel):
    results: List[Result]
    total_time: float
    parallel_faster_than_sequential: bool


async def calculate_square(number: int, delay: float) -> dict:
    start = time.perf_counter()
    await asyncio.sleep(delay)
    length = time.perf_counter() - start
    return {
        "number": number,
        "square": number ** 2,
        "delay": delay,
        "time": round(length, 2)}


@app.post("/calculate/", response_model=Response)
async def calculate(request: Request):
    if len(request.numbers) != len(request.delays):
        raise HTTPException(status_code=400, detail="Длины numbers и delays должны совпадать")

    parallel_start = time.perf_counter()

    tasks = [calculate_square(num, delay) for num, delay in zip(request.numbers, request.delays)]
    results = await asyncio.gather(*tasks)

    parallel_finish = time.perf_counter() - parallel_start
    sequential = sum(request.delays)

    return Response(
        results=[Result(**r) for r in results],
        total_time=round(parallel_finish, 2),
        parallel_faster_than_sequential=parallel_finish < sequential)


client = TestClient(app)

In [2]:
import json

response = client.post(
    "/calculate/",
    json={"numbers": [7, 11, 5], "delays": [2, 0.7, 0.3]})

print(json.dumps(response.json(), indent=2))

{
  "results": [
    {
      "number": 7,
      "square": 49,
      "delay": 2.0,
      "time": 2.0
    },
    {
      "number": 11,
      "square": 121,
      "delay": 0.7,
      "time": 0.7
    },
    {
      "number": 5,
      "square": 25,
      "delay": 0.3,
      "time": 0.3
    }
  ],
  "total_time": 2.0,
  "parallel_faster_than_sequential": true
}
